<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 vsep — Audio Stem Separator

Welcome to the **vsep** interactive demo! Separate any audio into vocals, drums, bass, instruments, and more using 100+ state-of-the-art AI models.

## Features
- ⚡ **Fast Processing** — GPU-accelerated inference with automatic hardware detection
- 🎯 **100+ Models** — Choose from Demucs, MDX-Net, VR Band Split, and Roformer architectures
- 🎚️ **Easy to Use** — Select, separate, listen, and download in 5 steps
- 📖 **[Full Wiki](https://github.com/BF667-IDLE/vsep/wiki)** for detailed docs

---

## 1️⃣ Setup

Clone the repo and install dependencies. This takes ~2 minutes.

In [ ]:
#@title 1. Install vsep

!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep
%cd /content/vsep
!pip install -q -r requirements.txt

import os
os.makedirs("output", exist_ok=True)
os.makedirs("models", exist_ok=True)

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️ No GPU detected — will use CPU (slower)")
    print("💡 Go to Runtime → Change runtime type → T4 GPU")

## 2️⃣ Upload Your Audio

Upload any audio file. Supported: MP3, WAV, FLAC, OGG, M4A, AAC, WMA

In [ ]:
#@title 2. Upload Audio

from google.colab import files
import os

print("📁 Upload your audio file...")
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
    size_mb = os.path.getsize(audio_file) / (1024 * 1024)
    print(f"✅ Uploaded: {audio_file} ({size_mb:.1f} MB)")
else:
    print("❌ No file uploaded.")

## 3️⃣ Select Model

Pick a model from the dropdown. The full catalog (189+ models) is loaded from both **models.json** and the **UVR model repo** — no typing required.

💡 You can **type in the dropdown** to search/filter models.

In [ ]:
#@title 3. Select Model

import sys
import builtins
sys.path.insert(0, "/content/vsep")

from separator import Separator
from config.variables import MODEL_CATALOG
from ipywidgets import Dropdown, HTML, VBox
from IPython.display import display

# ── Load dynamic models from UVR ─────────────────────────────────────
sep = Separator(info_only=True)
model_list = sep.list_models()

# ── Build flat catalog: display_name → filename ───────────────────────
# Start with local MODEL_CATALOG from models.json
catalog = dict(MODEL_CATALOG)

# Merge dynamic UVR models (covers VR, MDX, MDX23C, Roformer, Demucs)
for mtype, models in model_list.items():
    for name, info in models.items():
        catalog[name] = info["filename"]

# Sort alphabetically
sorted_names = sorted(catalog.keys(), key=lambda x: x.lower())
total = len(catalog)

# Count by architecture
arch_counts = {}
for mtype, models in model_list.items():
    arch_counts[mtype] = len(models)
counts_str = ", ".join(f"{k}: {v}" for k, v in sorted(arch_counts.items()))
print(f"  {total} models loaded  ({counts_str})")
print(f"  Type in the dropdown to search 🔍\n")

# ── Build ipywidgets dropdown ────────────────────────────────────────
default_name = next(
    (n for n in sorted_names if "BS Roformer" in n and "EP-317" in n),
    next((n for n in sorted_names if "htdemucs_ft" in n), sorted_names[0])
)

dropdown = Dropdown(
    options=sorted_names,
    value=default_name,
    description='',
    layout={'width': '95%'},
    rows=min(20, len(sorted_names)),
)

info_html = HTML("")

def on_select(change):
    name = change['new']
    fname = catalog[name]
    info_html.value = (
        f"<b>Selected:</b> {name}<br>"
        f"<b>File:</b> <code>{fname}</code>"
    )

dropdown.observe(on_select, names='value')
on_select({'new': default_name})

display(VBox([dropdown, info_html]))

# ── Store in builtins so the next cell can read the selection ─────────
builtins._vsep_catalog = catalog
builtins._vsep_dropdown = dropdown

In [ ]:
#@title 3b. Confirm Selection

selected_model = _vsep_catalog[_vsep_dropdown.value]
print(f"  ✅ {_vsep_dropdown.value}")
print(f"     → {selected_model}")

## 4️⃣ Separate Audio

Run the separation. First use may take a few minutes to download the model (cached afterwards).

In [ ]:
#@title 4. Run Separation

import sys
sys.path.insert(0, "/content/vsep")

from separator import Separator
import config.variables as cfg
import os, time

# Optimize for Colab
cfg.MAX_DOWNLOAD_WORKERS = 4
cfg.DOWNLOAD_CHUNK_SIZE = 262144

print("🚀 Starting separation...")
print(f"📁 Input: {audio_file}")
print(f"🎯 Model: {selected_model}")
print("⏳ Please wait...\n")

start = time.time()

# Initialize and run
separator = Separator(
    model_file_dir="./models",
    output_dir="./output",
    use_soundfile=True,
)

separator.load_model(selected_model)
output_files = separator.separate(audio_file)

elapsed = time.time() - start

print(f"\n✅ Separation complete! ({elapsed:.1f}s)")
print(f"📂 {len(output_files)} stem(s):")
for f in output_files:
    size = os.path.getsize(f) / (1024 * 1024)
    print(f"   — {os.path.basename(f)} ({size:.1f} MB)")

## 5️⃣ Listen to Results

Play back the separated stems.

In [ ]:
#@title 5. Listen to Stems

import glob
from IPython.display import Audio, display

output_files = sorted(glob.glob("output/*"))

if output_files:
    print(f"🎵 {len(output_files)} separated stem(s):\n")
    for i, fp in enumerate(output_files, 1):
        name = os.path.basename(fp)
        print(f"{i}. {name}")
        display(Audio(fp))
        print("─" * 60)
else:
    print("❌ No output files. Run the separation first.")

## 6️⃣ Download Results

Download the separated stems to your computer.

In [ ]:
#@title 6. Download Stems

from google.colab import files
import glob, os

output_files = sorted(glob.glob("output/*"))

if output_files:
    print(f"📥 Downloading {len(output_files)} file(s)...\n")
    for fp in output_files:
        files.download(fp)
else:
    print("❌ No files to download. Run the separation first.")

---

## 💡 Tips & Recommendations

### Best Models by Task
| Task | Recommended Model | Why |
|------|-----------------|-----|
| **Vocals** | `BS Roformer EP-317 sdr 12.9755` | Highest SDR (12.98) of any model |
| **4-stem** | `htdemucs_ft` | Vocals + drums + bass + other |
| **6-stem** | `htdemucs_6s` | Adds piano + guitar |
| **Instrumental** | `MelBand Roformer Kim Inst V1 Plus` | Clean instrumental |
| **Karaoke** | `MelBand Roformer Karaoke (becruily)` | Aggressive vocal removal |
| **De-Reverb** | `MelBand Roformer De-Reverb (anvuew)` | Remove reverb from vocals |
| **Denoise** | `Denoise Mel Roformer Aufr33` | Clean up background noise |
| **Fast** | `MDX-Net Inst HQ 5` | Lightweight, fast inference |

### GPU Tips
- **T4 (free)** — Good for most models. Big Roformer/Demucs may be tight on 16 GB
- **A100 (paid)** — Can run any model including the largest ones
- **Timeouts** — Sessions timeout after ~90 min of inactivity
- **Long files** — For audio > 30 min, add `chunk_duration=300` in the separation cell

### Troubleshooting

| Problem | Fix |
|---------|-----|
| **No GPU** | Runtime → Change runtime type → T4 GPU |
| **Out of memory** | Use a smaller model or shorter audio, or add `chunk_duration=300` |
| **Download failed** | Re-run the cell (downloads resume automatically) |
| **Module not found** | Re-run the Install cell or restart the runtime |

### Links

- 📦 **Repository:** [github.com/BF667-IDLE/vsep](https://github.com/BF667-IDLE/vsep)
- 📖 **Wiki:** [github.com/BF667-IDLE/vsep/wiki](https://github.com/BF667-IDLE/vsep/wiki)
- 🐛 **Issues:** [github.com/BF667-IDLE/vsep/issues](https://github.com/BF667-IDLE/vsep/issues)
- 📄 **License:** [MIT](https://opensource.org/licenses/MIT)

**Made with ❤️ using [vsep](https://github.com/BF667-IDLE/vsep)**